In [9]:
# 1. Обзор структуры данных
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data", 
                 header=None, 
                 names=["age", "workclass", "fnlwgt", "education", "education-num", 
                        "marital-status", "occupation", "relationship", "race", "sex", 
                        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"])

print("Признаки:", df.columns.tolist())
print("\nИнформация:")
df.info()
print("\nСтатистика:")
print(df.describe())

Признаки: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

Информация:
<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education-num   32561 non-null  int64
 5   marital-status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital-gain    32561 non-null  int64
 11  capital-loss    32561 non-null  int64
 12  hours-per-week  32561 non-null  int64
 13  native-country  32561 non-n

In [10]:
# 2. Обнаружение и обработка пропусков
print("Пропуски:\n", df.isna().sum())
# Удаляем строки с пропусками
df = df.replace(" ?", np.nan).dropna()
print("После удаления пропусков:", len(df))

Пропуски:
 age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64
После удаления пропусков: 30162


In [11]:
# 3. Обнаружение и удаление выбросов (IQR)
def remove_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[col] >= lower) & (data[col] <= upper)]

print("До:", len(df))
for col in ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]:
    df = remove_outliers(df, col)
print("После:", len(df))

До: 30162
После: 18458


In [12]:
# 4. Масштабирование числовых признаков
num_cols = df.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
print("Масштабирование выполнено")

Масштабирование выполнено


In [13]:
# 5. Кодирование категориальных признаков
# Label Encoding (порядковые)
le = LabelEncoder()
df["education"] = le.fit_transform(df["education"])

# One-Hot Encoding (номинальные)
df = pd.get_dummies(df, columns=["workclass", "marital-status", "occupation", 
                                  "relationship", "race", "sex", "native-country"], drop_first=True)

print("Кодирование выполнено")
print(df.head())

Кодирование выполнено
        age    fnlwgt  education  education-num  capital-gain  capital-loss  \
2 -0.019494  0.394143         10      -0.451122           0.0           0.0   
3  1.248639  0.612277          1      -1.289759           0.0           0.0   
4 -0.864915  1.798007          8       1.226150           0.0           0.0   
5 -0.104036  1.182465         11       1.645468           0.0           0.0   
7  1.164097  0.325484         10      -0.451122           0.0           0.0   

   hours-per-week  income  workclass_ Local-gov  workclass_ Private  ...  \
2       -0.378071   <=50K                 False                True  ...   
3       -0.378071   <=50K                 False                True  ...   
4       -0.378071   <=50K                 False                True  ...   
5       -0.378071   <=50K                 False                True  ...   
7        0.884583    >50K                 False               False  ...   

   native-country_ Portugal  native-country_ P

In [14]:
# 6. Финальный набор данных
print("Итоговый DataFrame:")
print(df.info())
print("\nПервые 5 строк:")
print(df.head())

Итоговый DataFrame:
<class 'pandas.DataFrame'>
Index: 18458 entries, 2 to 32558
Data columns (total 82 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   age                                         18458 non-null  float64
 1   fnlwgt                                      18458 non-null  float64
 2   education                                   18458 non-null  int64  
 3   education-num                               18458 non-null  float64
 4   capital-gain                                18458 non-null  float64
 5   capital-loss                                18458 non-null  float64
 6   hours-per-week                              18458 non-null  float64
 7   income                                      18458 non-null  str    
 8   workclass_ Local-gov                        18458 non-null  bool   
 9   workclass_ Private                          18458 non-null  bool   
 10  workcl